# Final Project - Image Segmentation and Matching in Computer Vision
city example

Computer Vision 2024-25 (M. Polo)

This project explores image segmentation with k-means method and matching 

In [ ]:
# Import the necessary packages
import numpy as np
import cv2 as cv
from matplotlib import pyplot as plt

In [ ]:
# Load images
city = cv.imread("../data/city.JPG")
pieces1 = cv.imread("../data/pieces1.JPG")
pieces2 = cv.imread("../data/pieces2.JPG")
pieces3 = cv.imread("../data/pieces3.JPG")
pz = cv.imread("../data/pz.jpg")

# I check if the images are loaded correctly
if city is None:
    print("Error: Could not load image 'peppawholeimage'")
else:
    # I convert the image to RGB
    city_rgb = cv.cvtColor(city, cv.COLOR_BGR2RGB)  # I convert to RGB for matplotlib

# Function to increase brightness
def increase_brightness(img, value=50):
    hsv = cv.cvtColor(img, cv.COLOR_RGB2HSV)  # I convert to HSV
    h, s, v = cv.split(hsv)
    v = np.clip(v + value, 0, 255)  # I increase brightness
    final_hsv = cv.merge((h, s, v))
    img_bright = cv.cvtColor(final_hsv, cv.COLOR_HSV2RGB)  # I convert back to RGB
    return img_bright


bright_city_rgb = increase_brightness(city_rgb, value=50)

plt.figure(figsize=(48, 24))
plt.subplot(2, 1, 1)
plt.imshow(city_rgb)
plt.title('Original Image')
plt.axis('off')

plt.subplot(2, 1, 2)
plt.imshow(bright_city_rgb)
plt.title('Brightened Image')
plt.axis('off')
plt.show()


- I proceed by increasing the brightness also on the pieces

In [ ]:
# convert to RGB for matplotlib
pieces1_rgb = cv.cvtColor(pieces1, cv.COLOR_BGR2RGB)
pieces2_rgb = cv.cvtColor(pieces2, cv.COLOR_BGR2RGB)
pieces3_rgb = cv.cvtColor(pieces3, cv.COLOR_BGR2RGB)

# I increase the brightness of pieces1-------------------------------
bright_pieces1_rgb = increase_brightness(pieces1_rgb, value=50)

plt.figure(figsize=(6, 12))
plt.subplot(2, 1, 1)  
plt.imshow(pieces1_rgb)
plt.title('pieces1 Original')
plt.axis('off')

plt.subplot(2, 1, 2)  
plt.imshow(bright_pieces1_rgb)
plt.title('pieces1 Brightened')
plt.axis('off')
plt.show()


# for pieces2-------------------------------------------------
bright_pieces2_rgb = increase_brightness(pieces2_rgb, value=50)

plt.figure(figsize=(6, 12))
plt.subplot(2, 1, 1) 
plt.imshow(pieces2_rgb)
plt.title('pieces2 Original')
plt.axis('off')

plt.subplot(2, 1, 2) 
plt.imshow(bright_pieces2_rgb)
plt.title('pieces2 Brightened')
plt.axis('off')
plt.show()

# for pieces3---------------------------------------------------
bright_pieces3_rgb = increase_brightness(pieces3_rgb, value=50)

plt.figure(figsize=(6, 12))
plt.subplot(2, 1, 1)
plt.imshow(pieces3_rgb)
plt.title('pieces3 Original')
plt.axis('off')

plt.subplot(2, 1, 2) 
plt.imshow(bright_pieces3_rgb)
plt.title('pieces3 Brightened')
plt.axis('off')
plt.show()


# K-MEANS Segmentation

let's begin with pieces1

In [ ]:
# ADAPTIVE RESIZING to speed up processing
resize_factor = 0.3
image_resized = cv.resize(bright_pieces1_rgb, (0, 0), fx=resize_factor, fy=resize_factor)

# I apply K-means to the image colors
k = 4  # number of clusters for K-means
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 100, 0.2)
pixel_values = image_resized.reshape((-1, 3)).astype(np.float32)
_, labels, centers = cv.kmeans(pixel_values, k, None, criteria, 10, cv.KMEANS_PP_CENTERS)

# I reconstruct the segmented image
segmented_image = centers[labels.flatten()].reshape(image_resized.shape).astype(np.uint8)

pixel_counts = [np.sum(labels == i) for i in range(k)]
sorted_clusters = np.argsort(pixel_counts)
clusters_to_include = sorted_clusters[:2]  # I include the clusters with fewer pixels

# I create the binary image by combining the selected clusters
binary_image = np.zeros(segmented_image.shape[:2], dtype=np.uint8)
for cluster in clusters_to_include:
    binary_image |= (labels.reshape(segmented_image.shape[:2]) == cluster)
binary_image = (binary_image * 255).astype(np.uint8)

# display the segmentation results
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1), plt.imshow(image_resized), plt.title('Original Image (resized)'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(segmented_image), plt.title('Segmented Image'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(binary_image, cmap='gray'), plt.title('Binary Image'), plt.axis('off')
plt.tight_layout()
plt.show()

# I detect contours on the binary image using CANNY
edges = cv.Canny(binary_image, 100, 200)

# perform dilation to fill internal edges
kernel = cv.getStructuringElement(cv.MORPH_RECT, (10, 10))
dilated = cv.dilate(edges, kernel, iterations=1)

# find the external contours
contours, _ = cv.findContours(dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
external_contour = np.zeros_like(image_resized)
cv.drawContours(external_contour, contours, -1, 255, thickness=cv.FILLED)

# perform erosion to separate the pieces
kernel = np.ones((10, 10), np.uint8)
eroded = cv.erode(external_contour, kernel, iterations=1)

#  display
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1), plt.imshow(image_resized), plt.title('Original Image'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(edges, cmap='gray'), plt.title('Edges (Canny)'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(eroded, cmap='gray'), plt.title('External Contour'), plt.axis('off')
plt.tight_layout()
plt.show()

# I separate the pieces using connectedComponents
eroded_gray = eroded if len(eroded.shape) == 2 else cv.cvtColor(eroded, cv.COLOR_BGR2GRAY)
num_labels, labels_connected = cv.connectedComponents(eroded_gray)

# I extract the pieces
pieces_1 = []
for label in range(1, num_labels):
    piece_mask = (labels_connected == label).astype(np.uint8) * 255
    piece = cv.bitwise_and(image_resized, image_resized, mask=piece_mask)
    component_size = np.sum(piece_mask)

    
    if component_size < 1000000 or component_size > 14000000:
        continue
    else:
        pieces_1.append(piece)
        # I display the extracted pieces
        plt.imshow(piece)
        plt.title(f"Piece {label}")
        plt.axis('off')
        plt.show()


pieces2

In [ ]:
# ADAPTIVE RESIZING to speed up processing
resize_factor = 0.3
image_resized = cv.resize(bright_pieces2_rgb, (0, 0), fx=resize_factor, fy=resize_factor)

# I apply K-means to the image colors
k = 4  # number of clusters for K-means
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 100, 0.2)
pixel_values = image_resized.reshape((-1, 3)).astype(np.float32)
_, labels, centers = cv.kmeans(pixel_values, k, None, criteria, 10, cv.KMEANS_PP_CENTERS)

# I reconstruct the segmented image
segmented_image = centers[labels.flatten()].reshape(image_resized.shape).astype(np.uint8)


pixel_counts = [np.sum(labels == i) for i in range(k)]
sorted_clusters = np.argsort(pixel_counts)
clusters_to_include = sorted_clusters[:2] 

# I create the binary image by combining the selected clusters
binary_image = np.zeros(segmented_image.shape[:2], dtype=np.uint8)
for cluster in clusters_to_include:
    binary_image |= (labels.reshape(segmented_image.shape[:2]) == cluster)
binary_image = (binary_image * 255).astype(np.uint8)

# display the segmentation results
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1), plt.imshow(image_resized), plt.title('Original Image (resized)'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(segmented_image), plt.title('Segmented Image'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(binary_image, cmap='gray'), plt.title('Binary Image'), plt.axis('off')
plt.tight_layout()
plt.show()

# I detect contours on the binary image using CANNY
edges = cv.Canny(binary_image, 100, 200)

# perform dilation to fill internal edges
kernel = cv.getStructuringElement(cv.MORPH_RECT, (10, 10))
dilated = cv.dilate(edges, kernel, iterations=1)

# find the external contours
contours, _ = cv.findContours(dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
external_contour = np.zeros_like(image_resized)
cv.drawContours(external_contour, contours, -1, 255, thickness=cv.FILLED)

# perform erosion to separate the pieces
kernel = np.ones((10, 10), np.uint8)
eroded = cv.erode(external_contour, kernel, iterations=1)

#  display
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1), plt.imshow(image_resized), plt.title('Original Image'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(edges, cmap='gray'), plt.title('Edges (Canny)'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(eroded, cmap='gray'), plt.title('External Contour'), plt.axis('off')
plt.tight_layout()
plt.show()

# I separate the pieces using connectedComponents
eroded_gray = eroded if len(eroded.shape) == 2 else cv.cvtColor(eroded, cv.COLOR_BGR2GRAY)
num_labels, labels_connected = cv.connectedComponents(eroded_gray)

# I extract the pieces
pieces_2 = []
for label in range(1, num_labels):
    piece_mask = (labels_connected == label).astype(np.uint8) * 255
    piece = cv.bitwise_and(image_resized, image_resized, mask=piece_mask)
    component_size = np.sum(piece_mask)

    if component_size < 1000000 or component_size > 12000000:
        continue
    else:
        pieces_2.append(piece)
        # Visualizza i pezzi estratti
        plt.imshow(piece)
        plt.title(f"Piece {label}")
        plt.axis('off')
        plt.show()

pieces3

In [ ]:
resize_factor = 0.3
image_resized = cv.resize(bright_pieces3_rgb, (0, 0), fx=resize_factor, fy=resize_factor)


k = 4 
criteria = (cv.TERM_CRITERIA_EPS + cv.TERM_CRITERIA_MAX_ITER, 100, 0.2)
pixel_values = image_resized.reshape((-1, 3)).astype(np.float32)
_, labels, centers = cv.kmeans(pixel_values, k, None, criteria, 10, cv.KMEANS_PP_CENTERS)


segmented_image = centers[labels.flatten()].reshape(image_resized.shape).astype(np.uint8)

pixel_counts = [np.sum(labels == i) for i in range(k)]
sorted_clusters = np.argsort(pixel_counts)
clusters_to_include = sorted_clusters[:2] 


binary_image = np.zeros(segmented_image.shape[:2], dtype=np.uint8)
for cluster in clusters_to_include:
    binary_image |= (labels.reshape(segmented_image.shape[:2]) == cluster)
binary_image = (binary_image * 255).astype(np.uint8)


plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1), plt.imshow(image_resized), plt.title('Original Image (resized)'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(segmented_image), plt.title('Segmented Image'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(binary_image, cmap='gray'), plt.title('Binary Image'), plt.axis('off')
plt.tight_layout()
plt.show()


edges = cv.Canny(binary_image, 100, 200)


kernel = cv.getStructuringElement(cv.MORPH_RECT, (10, 10))
dilated = cv.dilate(edges, kernel, iterations=1)


contours, _ = cv.findContours(dilated, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
external_contour = np.zeros_like(image_resized)
cv.drawContours(external_contour, contours, -1, 255, thickness=cv.FILLED)


kernel = np.ones((10, 10), np.uint8)
eroded = cv.erode(external_contour, kernel, iterations=1)


image_resized_rgb = cv.cvtColor(image_resized, cv.COLOR_BGR2RGB)


plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1), plt.imshow(image_resized), plt.title('Original Image'), plt.axis('off')
plt.subplot(1, 3, 2), plt.imshow(edges, cmap='gray'), plt.title('Edges (Canny)'), plt.axis('off')
plt.subplot(1, 3, 3), plt.imshow(eroded, cmap='gray'), plt.title('External Contour'), plt.axis('off')
plt.tight_layout()
plt.show()


eroded_gray = eroded if len(eroded.shape) == 2 else cv.cvtColor(eroded, cv.COLOR_BGR2GRAY)
num_labels, labels_connected = cv.connectedComponents(eroded_gray)


pieces_3 = []
for label in range(1, num_labels):
    piece_mask = (labels_connected == label).astype(np.uint8) * 255
    piece = cv.bitwise_and(image_resized, image_resized, mask=piece_mask)
    component_size = np.sum(piece_mask)

    
    if component_size < 1000000 or component_size > 12000000:
        continue
    else:
        pieces_3.append(piece)
        # Visualizza i pezzi estratti
        plt.imshow(piece)
        plt.title(f"Piece {label}")
        plt.axis('off')
        plt.show()

In [ ]:
# sum the segmented pieces
pieces = pieces_1 + pieces_2 + pieces_3


# Make use of sift in the reference image

In [ ]:
sift = cv.SIFT_create(nfeatures = 0, nOctaveLayers = 3, contrastThreshold = 0.04, edgeThreshold = 10, sigma = 1.6)
gray_image = cv.cvtColor(bright_city_rgb, cv.COLOR_BGR2GRAY)

# Finds the keypoints and descriptors for the image
img_keypoints, img_descriptors = sift.detectAndCompute(gray_image, None)

image_with_keypoints = cv.drawKeypoints(bright_city_rgb, img_keypoints, None)

plt.imshow(image_with_keypoints)
plt.title("Image with SIFT Keypoints")
plt.axis('off')
plt.show()


# print the number of keypoints
print(f"Numero di keypoints rilevati: {len(img_keypoints)}")
print(f"Dimensione dei descrittori: {img_descriptors.shape}")

# number of pieces
number_of_pieces = len(pieces_1+pieces_2+pieces_3)

print(f"Numero di elementi nella lista: {number_of_pieces}")


In [ ]:
sift = cv.SIFT_create(nfeatures = 0, nOctaveLayers = 3, contrastThreshold = 0.04, edgeThreshold = 10, sigma = 1.6)
gray_image = cv.cvtColor(bright_city_rgb, cv.COLOR_BGR2GRAY)

pieces = pieces_1 + pieces_2 + pieces_3  

pieces_keypoints = []
pieces_descriptors = []

for i, piece in enumerate(pieces): 
    keypoints, descriptors = sift.detectAndCompute(piece, None)
    pieces_keypoints.append(keypoints)
    pieces_descriptors.append(descriptors)
    
    
    piece_with_keypoints = cv.drawKeypoints(piece, keypoints, None)


if img_descriptors is not None:
    print(f"Numero di keypoints rilevati: {len(img_keypoints)}")
    print(f"Dimensione dei descrittori: {img_descriptors.shape}")
else:
    print("Errore: Nessun descrittore trovato nell'immagine principale.")


In [ ]:
# I find the matches between the descriptors of the image and those of the pieces like in PEPPA
ratio = 3  # Configurable Ratio Threshold for refining
matches = []

for i, p_descriptors in enumerate(pieces_descriptors):
    if p_descriptors is None or img_descriptors is None:
        print(f"Skipping piece {i+1} - No descriptors found.")
        continue
    
    # I find the matches using BFMatcher
    bf = cv.BFMatcher(cv.NORM_L2, crossCheck=True)
    matches_piece = bf.match(p_descriptors, img_descriptors)

    # I filter out matches with distance 0 (potentially incorrect)
    filtered_matches = [match for match in matches_piece if match.distance > 0]

    # I refine the matches using the ratio threshold
    if filtered_matches:
        min_distance = min([match.distance for match in filtered_matches])
        refined_matches = [match for match in matches_piece if match.distance < ratio * min_distance]
        matches.append(refined_matches)

        # I display the refined matches
        matches_image = cv.drawMatches(pieces[i], pieces_keypoints[i], bright_city_rgb, img_keypoints, refined_matches, None)
        plt.figure(figsize=(15, 15))
        plt.imshow(matches_image)
        plt.title(f"Refined Matches for Piece {i + 1}")
        plt.axis('off')
        plt.show()
    else:
        matches.append([])
        print(f"No valid matches for piece {i+1}")


# wanna know the distances between corresponding points in the matches between puzzle pieces and the original image to understand how can i use them


In [ ]:
# distances between corresponding points
for i, refined_matches in enumerate(matches):
    if len(refined_matches) >= 3:
        piece_points = np.float32([pieces_keypoints[i][match.queryIdx].pt for match in refined_matches])
        image_points = np.float32([img_keypoints[match.trainIdx].pt for match in refined_matches])
        
        # Euclidean distances
        distances = np.linalg.norm(piece_points - image_points, axis=1)
        
        # I display the distances for each match
        print(f"Distances between matches for piece {i+1}:")
        print(distances)
        
        # I calculate the mean distance and standard deviation
        mean_distance = np.mean(distances)
        std_distance = np.std(distances)
        print(f"Mean Distance: {mean_distance:.2f}, Standard Deviation: {std_distance:.2f}")

        ''' remove to see the plots
        # I display a graph of the distances
        plt.figure(figsize=(10, 6))
        plt.plot(distances, marker='o', linestyle='-', color='blue', label='Distance')
        plt.axhline(mean_distance, color='red', linestyle='--', label='Mean')
        plt.axhline(mean_distance + std_distance, color='green', linestyle='--', label='Mean + Std')
        plt.axhline(mean_distance - std_distance, color='green', linestyle='--', label='Mean - Std')
        plt.title(f"Distances between Matches for Piece {i+1}")
        plt.xlabel("Match Index")
        plt.ylabel("Euclidean Distance")
        plt.legend()
        plt.grid(True)
        plt.show()
        '''

In [ ]:
# I calculate the Affine Transformation using the refined matches like in peppa
affine_transforms = []
canvas = np.zeros((bright_city_rgb.shape[0], bright_city_rgb.shape[1], 3), dtype=np.uint8)  # RGB, no alpha

for i, refined_matches in enumerate(matches):
    if len(refined_matches) >= 3:  # At least 3 matches are needed for an affine transformation
        piece_points = np.float32([pieces_keypoints[i][match.queryIdx].pt for match in refined_matches]).reshape(-1, 1, 2)
        image_points = np.float32([img_keypoints[match.trainIdx].pt for match in refined_matches]).reshape(-1, 1, 2)

        
        distances = np.linalg.norm(piece_points - image_points, axis=1)
        
       
        variance = np.var(distances)
        print(f"Piece {i+1}: Variance of distances = {variance:.2f}")

        # I calculate the affine transformation using the refined matches
        transform, inliers = cv.estimateAffinePartial2D(
            piece_points, 
            image_points, 
            method=cv.RANSAC, 
            ransacReprojThreshold=3.0
        )

        if transform is not None:
            affine_transforms.append(transform)

            # I apply the affine transformation to the piece
            transformed_piece = cv.warpAffine(
                pieces[i], 
                transform, 
                (bright_city_rgb.shape[1], bright_city_rgb.shape[0])
            )

            # I create a binary mask of the transformed piece
            mask = cv.cvtColor(transformed_piece, cv.COLOR_BGR2GRAY)
            _, mask = cv.threshold(mask, 1, 255, cv.THRESH_BINARY)

            # I apply the mask to overlay the transformed piece on the canvas
            canvas[mask > 0] = transformed_piece[mask > 0]

            # display
            print(f"Transformation matrix for piece {i + 1}:")
            print(transform)
            plt.figure(figsize=(6, 6))
            plt.imshow(transformed_piece)
            plt.title(f"Piece {i+1} Transformed with Refined Matches")
            plt.axis("off")
            plt.show()
        else:
            print(f"No valid transformation found for piece {i + 1}")
    else:
        print(f"Not enough matches for piece {i + 1} to compute affine transformation.")


plt.figure(figsize=(10, 6))
plt.imshow(canvas)
plt.title("Puzzle Reconstructed with Refined Affine Transformation")
plt.axis("off")
plt.show()
